# NGS Pipeline Stepwise Analysis

This notebook walks through the existing NGS pipeline outputs script by script and visualizes what happened at each step.

Scope:
- Analysis only: it reads the files produced by the pipeline scripts.
- Stepwise: preprocessing, UMI consensus, singleton rescue, filtering/extraction, and variant labeling.
- QC + interpretation: each section includes summaries and plots that are useful for debugging and presentation.


## 1. Environment Setup and Imports

Load the analysis stack, point at the repository, and define display defaults for notebook use.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

REPO_ROOT = Path("/cluster/project/reddy/katja/NGS_pipeline")
NOTEBOOK_ROOT = REPO_ROOT / "notebooks"
SCRIPT_ROOT = REPO_ROOT / "scripts"
DATA_ROOT = REPO_ROOT / "data"
RESULTS_ROOT = REPO_ROOT / "results"

print(f"Repository root: {REPO_ROOT}")
print(f"Notebook root:   {NOTEBOOK_ROOT}")
print(f"Data root:       {DATA_ROOT}")
print(f"Results root:    {RESULTS_ROOT}")

## 2. Define Configuration and Constants

Create a small configuration block so the notebook can discover the existing outputs without hard-coding stage-specific values throughout the analysis.

In [ ]:
CONFIG = {
    "sample_sheet": DATA_ROOT / "sample_sheet.tsv",
    "preprocessing_output": REPO_ROOT / "ml" / "results",
    "consensus_output": REPO_ROOT / "ml" / "results",
    "singleton_rescue_output": REPO_ROOT / "ml" / "results",
    "extraction_output": REPO_ROOT / "ml" / "results",
    "variant_labeling_output": REPO_ROOT / "ml" / "results",
}

PATTERNS = {
    "preprocessing_summary": "*.bbduk_summary.csv",
    "preprocessing_run_summary": "bbduk_run_summary_*.csv",
    "consensus_qc": "*_consensus_qc.tsv",
    "consensus_summary": "*_umi_summary.txt",
    "singletons": "*_singletons.fastq.gz",
    "singletons_rescued": "*_singletons_rescued.fastq.gz",
    "singleton_variants": "*_singleton_variants_*umis.tsv",
    "extraction_summary": "*.filtered.summary.tsv",
    "extraction_aa": "*.filtered.PASS.aa.tsv",
    "variant_labeling": "*.variant_labeling.csv",
    "variant_labeling_summary": "variant_labeling_summary.csv",
    "metrics": "per_sample_metrics.tsv",
}

print("Configured input patterns:")
for key, value in PATTERNS.items():
    print(f"  {key}: {value}")

## 3. Create Core Function Skeleton

Define reusable helpers for file discovery, metrics loading, and stage-specific summary parsing.

In [ ]:
SEARCH_ROOTS = [
    REPO_ROOT / "ml" / "results",
    REPO_ROOT / "results",
    REPO_ROOT / "NGS_pipeline" / "results",
    REPO_ROOT / "NGS_pipeline" / "data",
]


def discover_files(pattern: str, roots: Iterable[Path] = SEARCH_ROOTS) -> List[Path]:
    matches: List[Path] = []
    for root in roots:
        if root.exists():
            matches.extend(sorted(root.rglob(pattern)))
    unique: List[Path] = []
    seen = set()
    for path in matches:
        if path not in seen:
            unique.append(path)
            seen.add(path)
    return unique


def load_tabular(path: Path, sep: str = ",") -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, sep=sep)


def load_metrics_table(metrics_path: Path) -> pd.DataFrame:
    if not metrics_path.exists():
        return pd.DataFrame(columns=["sample_id", "sample_name", "step", "metric", "value"])
    return pd.read_csv(metrics_path, sep="\t")


def summarize_metrics(metrics: pd.DataFrame, step: str) -> pd.DataFrame:
    if metrics.empty:
        return metrics.copy()
    return metrics.loc[metrics["step"] == step].copy()


def parse_sectioned_summary_tsv(path: Path) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    lines = path.read_text().splitlines()
    metric_rows: List[Tuple[str, str]] = []
    strategy_rows: List[Tuple[str, str]] = []
    strategy_header: Optional[str] = None
    in_metric_section = False
    in_strategy_section = False

    for raw in lines:
        line = raw.strip()
        if not line:
            continue
        if line.startswith("# strategy="):
            in_strategy_section = True
            in_metric_section = False
            continue
        if line.startswith("#"):
            continue
        if line == "metric\tvalue":
            in_metric_section = True
            in_strategy_section = False
            continue
        if line in {"fail_position_1based\tcount", "mutations_per_read\tcount", "mutated_position_1based\tcount"}:
            strategy_header = line
            in_metric_section = False
            in_strategy_section = True
            continue
        parts = line.split("\t")
        if in_metric_section and len(parts) >= 2:
            metric_rows.append((parts[0], parts[1]))
        elif in_strategy_section and len(parts) >= 2:
            strategy_rows.append((parts[0], parts[1]))

    metric_df = pd.DataFrame(metric_rows, columns=["metric", "value"])
    if not metric_df.empty:
        metric_df["value"] = pd.to_numeric(metric_df["value"], errors="coerce")

    strategy_df: Optional[pd.DataFrame]
    if strategy_rows and strategy_header:
        col_a, col_b = strategy_header.split("\t")
        strategy_df = pd.DataFrame(strategy_rows, columns=[col_a, col_b])
        strategy_df[col_b] = pd.to_numeric(strategy_df[col_b], errors="coerce")
    else:
        strategy_df = None

    return metric_df, strategy_df


def add_rate_columns(df: pd.DataFrame, numerator: str, denominator: str, out_col: str) -> pd.DataFrame:
    out = df.copy()
    out[out_col] = np.where(out[denominator] > 0, out[numerator] / out[denominator], np.nan)
    return out


def show_file_inventory(title: str, paths: List[Path], limit: int = 10) -> None:
    print(title)
    if not paths:
        print("  no matches found")
        return
    for path in paths[:limit]:
        print(f"  {path}")
    if len(paths) > limit:
        print(f"  ... {len(paths) - limit} more")

## 4. Implement Input Validation

Validate that the expected input files exist and that the loaded tables contain the columns needed for plotting and summarization.

In [ ]:
def require_columns(df: pd.DataFrame, columns: Iterable[str], label: str) -> None:
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def validate_not_empty(df: pd.DataFrame, label: str) -> None:
    if df.empty:
        raise ValueError(f"{label} is empty")


def validate_path_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def validate_stage_inputs() -> Dict[str, List[Path]]:
    stage_inputs = {
        "metrics": discover_files(PATTERNS["metrics"]),
        "preprocessing": discover_files(PATTERNS["preprocessing_summary"]),
        "consensus": discover_files(PATTERNS["consensus_qc"]),
        "singleton_rescue": discover_files(PATTERNS["singleton_variants"]),
        "extraction": discover_files(PATTERNS["extraction_summary"]),
        "variant_labeling": discover_files(PATTERNS["variant_labeling"]),
    }
    for stage, paths in stage_inputs.items():
        print(f"{stage}: {len(paths)} file(s)")
    return stage_inputs


stage_inputs = validate_stage_inputs()

## 5. Add Main Execution Flow

Load the pipeline outputs stage by stage and generate the core QC and interpretation plots for each script.

In [ ]:
def plot_bar(df: pd.DataFrame, x: str, y: str, title: str, xlabel: str, ylabel: str, rotation: int = 45) -> None:
    if df.empty:
        print(f"Skipping plot: {title}")
        return
    ax = df.plot(kind="bar", x=x, y=y, legend=False, color="#2a6f97")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotation)
    plt.tight_layout()
    plt.show()


def plot_hist(series: pd.Series, title: str, xlabel: str, bins: int = 30) -> None:
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        print(f"Skipping plot: {title}")
        return
    plt.figure()
    sns.histplot(values, bins=bins, kde=True, color="#2a9d8f")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


def plot_scatter(df: pd.DataFrame, x: str, y: str, title: str, xlabel: str, ylabel: str) -> None:
    if df.empty:
        print(f"Skipping plot: {title}")
        return
    plt.figure()
    sns.scatterplot(data=df, x=x, y=y, s=70, color="#e76f51")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def plot_heatmap(df: pd.DataFrame, title: str, xlabel: str, ylabel: str) -> None:
    if df.empty:
        print(f"Skipping plot: {title}")
        return
    plt.figure(figsize=(12, 5))
    sns.heatmap(df, cmap="viridis", linewidths=0.2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def run_stepwise_analysis() -> Dict[str, pd.DataFrame]:
    outputs: Dict[str, pd.DataFrame] = {}

    print("=== Global metrics overview ===")
    metrics_files = discover_files(PATTERNS["metrics"])
    if metrics_files:
        metrics_frames = []
        for path in metrics_files:
            df = load_metrics_table(path)
            if not df.empty:
                df = df.copy()
                df["metrics_source"] = path.name
                metrics_frames.append(df)
        metrics = pd.concat(metrics_frames, ignore_index=True) if metrics_frames else pd.DataFrame(columns=["sample_id", "sample_name", "step", "metric", "value", "metrics_source"])
    else:
        metrics = pd.DataFrame(columns=["sample_id", "sample_name", "step", "metric", "value", "metrics_source"])
    outputs["metrics"] = metrics
    if metrics.empty:
        print("No per_sample_metrics.tsv files found")
    else:
        display(metrics.head())
        print(metrics["step"].value_counts().sort_index())

    print("\n=== Script 1: preprocessing ===")
    bbduk_files = discover_files(PATTERNS["preprocessing_summary"])
    run_summaries = discover_files(PATTERNS["preprocessing_run_summary"])
    show_file_inventory("BBDuk summary files:", bbduk_files)
    show_file_inventory("Run-level summary files:", run_summaries)
    if bbduk_files:
        bbduk_tables = []
        for path in bbduk_files:
            df = load_tabular(path, sep=",")
            require_columns(df, ["sample_id", "sample_name", "reads_in", "reads_out"], f"BBDuk summary {path.name}")
            df = df.copy()
            df["sample_label"] = df["sample_id"].astype(str) + "_" + df["sample_name"].astype(str)
            df = add_rate_columns(df, "reads_out", "reads_in", "retention_rate")
            bbduk_tables.append(df)
        bbduk = pd.concat(bbduk_tables, ignore_index=True)
        outputs["preprocessing"] = bbduk
        display(bbduk[["sample_label", "reads_in", "reads_out", "retention_rate", "runtime_s"]].head())
        plot_bar(
            bbduk.sort_values("retention_rate", ascending=False),
            x="sample_label",
            y="retention_rate",
            title="Script 1: Read retention per sample",
            xlabel="Sample",
            ylabel="Retention rate",
        )
        plot_scatter(
            bbduk,
            x="reads_in",
            y="reads_out",
            title="Script 1: Input vs output reads",
            xlabel="Reads in",
            ylabel="Reads out",
        )

    print("\n=== Script 2: UMI consensus ===")
    consensus_qc_files = discover_files(PATTERNS["consensus_qc"])
    consensus_summary_files = discover_files(PATTERNS["consensus_summary"])
    show_file_inventory("Consensus QC files:", consensus_qc_files)
    show_file_inventory("Consensus summary files:", consensus_summary_files)
    if consensus_qc_files:
        consensus_tables = []
        for path in consensus_qc_files:
            df = load_tabular(path, sep="\t")
            require_columns(df, ["umi", "n_reads", "median_log_delta"], f"Consensus QC {path.name}")
            df = df.copy()
            df["sample_label"] = path.name.replace("_consensus_qc.tsv", "")
            df["sample_label"] = df["sample_label"].str.replace("_", " ", regex=False)
            consensus_tables.append(df)
        consensus = pd.concat(consensus_tables, ignore_index=True)
        outputs["consensus_qc"] = consensus
        display(consensus[["umi", "n_reads", "median_log_delta", "sample_label"]].head())
        plot_hist(
            consensus["median_log_delta"],
            title="Script 2: Consensus confidence (median log delta)",
            xlabel="Median log delta",
        )
        if metrics.empty is False:
            step2 = summarize_metrics(metrics, "02_umi_consensus")
            if not step2.empty:
                pivot = step2.pivot_table(index=["sample_id", "sample_name"], columns="metric", values="value", aggfunc="first").reset_index()
                pivot["sample_label"] = pivot["sample_id"].astype(str) + "_" + pivot["sample_name"].astype(str)
                cols = [c for c in ["reads_in_consensus", "reads_in_singletons", "umis_consensus", "umis_singleton"] if c in pivot.columns]
                if cols:
                    display(pivot[["sample_label"] + cols].head())
                if {"reads_in_consensus", "reads_in_singletons"}.issubset(pivot.columns):
                    long = pivot[["sample_label", "reads_in_consensus", "reads_in_singletons"]].melt(id_vars="sample_label", var_name="class", value_name="reads")
                    plt.figure(figsize=(12, 5))
                    sns.barplot(data=long, x="sample_label", y="reads", hue="class")
                    plt.title("Script 2: Reads contributing to consensus vs singletons")
                    plt.xlabel("Sample")
                    plt.ylabel("Reads")
                    plt.xticks(rotation=45, ha="right")
                    plt.tight_layout()
                    plt.show()

    print("\n=== Script 2a: singleton rescue ===")
    singleton_variants = discover_files(PATTERNS["singleton_variants"])
    rescued_fastqs = discover_files(PATTERNS["singletons_rescued"])
    show_file_inventory("Singleton variant tables:", singleton_variants)
    show_file_inventory("Rescued singleton FASTQs:", rescued_fastqs)
    if singleton_variants:
        variant_tables = []
        for path in singleton_variants:
            df = load_tabular(path, sep="\t")
            require_columns(df, ["variant_33bp", "n_umis"], f"Singleton variant table {path.name}")
            df = df.copy()
            df["sample_label"] = path.name.replace("_singleton_variants_", "|")
            variant_tables.append(df)
        variants = pd.concat(variant_tables, ignore_index=True)
        outputs["singleton_variants"] = variants
        display(variants.head())
        plot_hist(
            variants["n_umis"],
            title="Script 2a: UMI support per rescued variant",
            xlabel="n_umis",
            bins=20,
        )

    if metrics.empty is False:
        step2a = summarize_metrics(metrics, "02a_singleton_rescue")
        if not step2a.empty:
            pivot = step2a.pivot_table(index=["sample_id", "sample_name"], columns="metric", values="value", aggfunc="first").reset_index()
            pivot["sample_label"] = pivot["sample_id"].astype(str) + "_" + pivot["sample_name"].astype(str)
            if {"reads_rescued", "reads_in", "variants_kept"}.issubset(pivot.columns):
                display(pivot[["sample_label", "reads_in", "reads_rescued", "variants_kept"]].head())
                plot_bar(
                    pivot.sort_values("reads_rescued", ascending=False),
                    x="sample_label",
                    y="reads_rescued",
                    title="Script 2a: Reads rescued after singleton filtering",
                    xlabel="Sample",
                    ylabel="Reads rescued",
                )

    print("\n=== Script 3: library filtering and extraction ===")
    extraction_summaries = discover_files(PATTERNS["extraction_summary"])
    extraction_aa = discover_files(PATTERNS["extraction_aa"])
    show_file_inventory("Extraction summaries:", extraction_summaries)
    show_file_inventory("PASS AA tables:", extraction_aa)
    extraction_metric_frames = []
    extraction_strategy_frames = []
    for path in extraction_summaries:
        metric_df, strategy_df = parse_sectioned_summary_tsv(path)
        if not metric_df.empty:
            metric_df = metric_df.copy()
            metric_df["sample_label"] = path.name.replace(".filtered.summary.tsv", "")
            extraction_metric_frames.append(metric_df)
        if strategy_df is not None and not strategy_df.empty:
            strategy_df = strategy_df.copy()
            strategy_df["sample_label"] = path.name.replace(".filtered.summary.tsv", "")
            extraction_strategy_frames.append(strategy_df)
    if extraction_metric_frames:
        extraction_metrics = pd.concat(extraction_metric_frames, ignore_index=True)
        outputs["extraction_metrics"] = extraction_metrics
        display(extraction_metrics.head())
        if set(extraction_metrics["metric"]).intersection({"pass", "fail_anchor_not_found", "fail_region_too_short", "fail_contains_N"}):
            summary_wide = extraction_metrics.pivot_table(index="sample_label", columns="metric", values="value", aggfunc="first").fillna(0)
            for column in ["pass", "fail_anchor_not_found", "fail_region_too_short", "fail_contains_N", "fail_wrong_length", "fail_stop_codon", "fail_degenerate_mismatch", "fail_too_many_mutations", "fail_mutation_not_NNK"]:
                if column not in summary_wide.columns:
                    summary_wide[column] = 0
            summary_wide = summary_wide.reset_index()
            plot_cols = [column for column in ["pass", "fail_anchor_not_found", "fail_region_too_short", "fail_contains_N", "fail_wrong_length", "fail_stop_codon", "fail_degenerate_mismatch", "fail_too_many_mutations", "fail_mutation_not_NNK"] if summary_wide[column].sum() > 0]
            if plot_cols:
                long = summary_wide[["sample_label"] + plot_cols].melt(id_vars="sample_label", var_name="metric", value_name="count")
                plt.figure(figsize=(14, 5))
                sns.barplot(data=long, x="sample_label", y="count", hue="metric")
                plt.title("Script 3: Extraction pass/fail breakdown")
                plt.xlabel("Sample")
                plt.ylabel("Count")
                plt.xticks(rotation=45, ha="right")
                plt.tight_layout()
                plt.show()
    if extraction_strategy_frames:
        strategy_all = pd.concat(extraction_strategy_frames, ignore_index=True)
        outputs["extraction_strategy"] = strategy_all
        display(strategy_all.head())
        strategy_columns = [column for column in strategy_all.columns if column not in {"sample_label"}]
        if len(strategy_columns) >= 2:
            key_col = strategy_columns[0]
            value_col = strategy_columns[1]
            strategy_plot = strategy_all.groupby(key_col, as_index=False)[value_col].sum().sort_values(value_col, ascending=False)
            plot_bar(
                strategy_plot,
                x=key_col,
                y=value_col,
                title="Script 3: Strategy-specific summary",
                xlabel=key_col,
                ylabel=value_col,
            )
    if extraction_aa:
        aa_tables = []
        for path in extraction_aa[:10]:
            df = load_tabular(path, sep="\t")
            require_columns(df, ["read_id", "nt_seq", "aa_seq"], f"PASS AA table {path.name}")
            df = df.copy()
            df["sample_label"] = path.name.replace(".filtered.PASS.aa.tsv", "")
            df["aa_len"] = df["aa_seq"].astype(str).str.len()
            aa_tables.append(df)
        aa = pd.concat(aa_tables, ignore_index=True)
        outputs["extraction_aa"] = aa
        display(aa[["sample_label", "aa_len"]].head())
        plot_hist(aa["aa_len"], title="Script 3: Amino-acid sequence length", xlabel="AA length", bins=15)

    print("\n=== Script 4: variant labeling ===")
    variant_label_files = discover_files(PATTERNS["variant_labeling"])
    variant_summary_files = discover_files(PATTERNS["variant_labeling_summary"])
    show_file_inventory("Variant labeling CSVs:", variant_label_files)
    show_file_inventory("Variant labeling summary files:", variant_summary_files)
    if variant_summary_files:
        variant_summary = load_tabular(variant_summary_files[0], sep=",")
        outputs["variant_summary"] = variant_summary
        display(variant_summary.head())
        if {"n_spec_0", "n_spec_1", "n_spec_2"}.issubset(variant_summary.columns):
            spec_long = variant_summary[["peptide", "n_spec_0", "n_spec_1", "n_spec_2"]].melt(id_vars="peptide", var_name="class", value_name="count")
            plt.figure(figsize=(12, 5))
            sns.barplot(data=spec_long, x="peptide", y="count", hue="class")
            plt.title("Script 4: Specificity class counts per peptide")
            plt.xlabel("Peptide")
            plt.ylabel("Variants")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()
    if variant_label_files:
        first_variant = load_tabular(variant_label_files[0], sep=",")
        outputs["variant_example"] = first_variant
        display(first_variant.head())
        enrich_col = "enrich_pos3x_vs_neg"
        if enrich_col in first_variant.columns:
            plot_hist(first_variant[enrich_col], title="Script 4: Enrichment distribution", xlabel=enrich_col, bins=30)
        if "specificity" in first_variant.columns:
            spec_counts = first_variant["specificity"].value_counts().sort_index().reset_index()
            spec_counts.columns = ["specificity", "count"]
            plot_bar(
                spec_counts,
                x="specificity",
                y="count",
                title="Script 4: Specificity classes in one peptide table",
                xlabel="Specificity class",
                ylabel="Count",
                rotation=0,
            )

    print("\n=== Combined summary ===")
    if outputs:
        print("Loaded stages:", ", ".join(outputs.keys()))
    else:
        print("No stage outputs loaded yet.")

    return outputs


analysis_outputs = run_stepwise_analysis()

## 6. Write Basic Unit Tests

Run a few minimal tests against the helper functions and parsers so the notebook can be checked without the full dataset.

In [ ]:
from tempfile import TemporaryDirectory


def run_basic_tests() -> None:
    sample = pd.DataFrame({"reads_in": [10, 0], "reads_out": [5, 0], "sample_name": ["a", "b"]})
    with_rate = add_rate_columns(sample, "reads_out", "reads_in", "retention_rate")
    assert math.isclose(with_rate.loc[0, "retention_rate"], 0.5)
    assert np.isnan(with_rate.loc[1, "retention_rate"])

    require_columns(with_rate, ["reads_in", "reads_out"], "sample")
    try:
        require_columns(with_rate, ["missing_col"], "sample")
        raise AssertionError("Expected require_columns to raise")
    except ValueError:
        pass

    with TemporaryDirectory() as tmpdir:
        tmp_path = Path(tmpdir) / "summary.tsv"
        tmp_path.write_text(
            "# inputs\n"
            "# fake.tsv\n"
            "\n"
            "metric\tvalue\n"
            "pass\t12\n"
            "fail_anchor_not_found\t3\n"
            "\n"
            "# strategy=3xNNK\n"
            "mutations_per_read\tcount\n"
            "0\t7\n"
            "1\t5\n"
        )
        metric_df, strategy_df = parse_sectioned_summary_tsv(tmp_path)
        assert list(metric_df["metric"]) == ["pass", "fail_anchor_not_found"]
        assert list(metric_df["value"]) == [12, 3]
        assert strategy_df is not None
        assert list(strategy_df.columns) == ["mutations_per_read", "count"]
        assert list(strategy_df["count"]) == [7, 5]

    print("Basic notebook tests passed")


run_basic_tests()